# Train and persist the production model

Fits the EBM on the full dataset, fits the gate fallbacks, and saves
`aurora_ebm_model_final_deployment.joblib`. See `aurora_gated_model.py` for the model architecture.

In [1]:
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from interpret.glassbox import ExplainableBoostingClassifier

from aurora_gated_model import GatedAuroraModel
from aurora_transforms import sine_mlon, cosine_mlon

DATA_CSV = "aurora_dataset_clean_timestart.csv"
MODEL_OUT = "aurora_ebm_model_final_deployment.joblib"

## Load, clean, and label the data

In [2]:
data = pd.read_csv(DATA_CSV)

day_of_year = pd.to_datetime(data["time_start"]).dt.dayofyear
declination = np.radians(23.45 * np.sin(np.radians(360 / 365 * (day_of_year - 81))))
lat_rad = np.radians(data["latitude"])
data["solar_noon_elev_max"] = np.degrees(
    np.arcsin(np.sin(lat_rad) * np.sin(declination) + np.cos(lat_rad) * np.cos(declination))
)
polar_winter = data["solar_noon_elev_max"] <= 6
dark_enough = (data["sun_elevation_localtime"] <= 6) | (data["sun_elevation_utc"] <= 6) | (data["kp"] >= 7)

features0 = ["mlat", "mlon", "kp", "bz_gsm", "solar_wind_speed", "oval_dist", "Ap"]
features1primary0 = ["mlat", "mlon", "kp", "bz_gsm", "solar_wind_speed"]
features2primary = ["cloud_cover", "moon_darkness", "sun_elevation_utc"]
featureslow = ["mlat", "mlon", "kp", "bz_gsm", "solar_wind_speed", "oval_dist", "Ap",
               "cloud_cover", "moon_darkness", "sun_elevation_utc"]
target = "see_aurora"

dataclean = data.dropna(subset=features1primary0, how="any").copy()
sighting = dataclean[["see_aurora"]].copy()
sighting["see_aurora"] = (dataclean["see_aurora"] == True) & (polar_winter | dark_enough)
dataclean["see_aurora"] = sighting["see_aurora"]

mask_valid = (
    (dataclean["solar_wind_speed"] < 9000) & (dataclean["bz_gsm"].abs() < 900)
    & (dataclean["sigma_bz_gsm"] < 900) & (dataclean["kp"] < 90)
    & (dataclean["ap"] < 900) & (dataclean["Ap"] < 900)
)
dataclean = dataclean[mask_valid].copy()

dataclean["oval_dist"] = dataclean["mlat"].abs() - (66.0 - 2.0 * dataclean["kp"])
dataclean["mlat_band"] = pd.cut(
    dataclean["mlat"].abs(), bins=[0, 48, 63, 90], labels=["00-48", "48-63", "63-90"]
)
print(f"Rows: {len(dataclean):,}")

Rows: 22,174


## Pipeline and per-band class-balance weights

In [3]:
feat_names = ["sin_mlon", "cos_mlon"] + [f for f in features0 if f != "mlon"] + features2primary
interactions = [(3, 4), (3, 5), (4, 5)]  # (kp,bz_gsm), (kp,solar_wind_speed), (bz_gsm,solar_wind_speed)

preprocess = ColumnTransformer(
    transformers=[
        ("sine_mlon", FunctionTransformer(sine_mlon), ["mlon"]),
        ("cosine_mlon", FunctionTransformer(cosine_mlon), ["mlon"]),
    ],
    remainder="passthrough",
)
baseline_pipeline = Pipeline(steps=[
    ("prepro", preprocess),
    ("EBM", ExplainableBoostingClassifier(
        feature_names=feat_names, interactions=interactions,
        max_bins=128, max_rounds=1000, min_samples_leaf=5, random_state=42,
    )),
])


def per_band_weights(y, bands):
    y = np.array(y)
    bands = np.array(bands)
    weights = np.ones(len(y), dtype=float)
    for band in np.unique(bands[~pd.isnull(bands)]):
        mask = bands == band
        y_b = y[mask]
        n_neg = (y_b == False).sum()
        n_pos = (y_b == True).sum()
        n_tot = len(y_b)
        if n_neg > 0:
            weights[mask & (y == False)] = n_tot / (2 * n_neg)
        if n_pos > 0:
            weights[mask & (y == True)] = n_tot / (2 * n_pos)
    return weights


y = dataclean[target].astype(int)
sample_weight = per_band_weights(y, dataclean["mlat_band"])
X = dataclean[featureslow]

baseline_pipeline.fit(X, y, EBM__sample_weight=sample_weight)
print("EBM fit.")

EBM fit.


## Gate fallbacks (`oval_dist`/`|mlat|` and `sun_elevation_utc`)

In [4]:
lr_oval_input = dataclean[["oval_dist"]].copy()
lr_oval_input["abs_mlat"] = dataclean["mlat"].abs()
lr_oval = LogisticRegression()
lr_oval.fit(lr_oval_input, y, sample_weight=sample_weight)

neg_mask = dataclean["sun_elevation_utc"] < 0
lr_sun_neg = LogisticRegression()
lr_sun_neg.fit(dataclean.loc[neg_mask, ["sun_elevation_utc"]], y[neg_mask.values], sample_weight=sample_weight[neg_mask.values])

pos_mask = dataclean["sun_elevation_utc"] >= 0
lr_sun_pos = LogisticRegression()
lr_sun_pos.fit(dataclean.loc[pos_mask, ["sun_elevation_utc"]], y[pos_mask.values], sample_weight=sample_weight[pos_mask.values])

print("Gate fallbacks fit.")

Gate fallbacks fit.


## Assemble and persist

In [5]:
model = GatedAuroraModel(
    baseline_pipeline, lr_oval, featureslow,
    lr_features=["oval_dist", "abs_mlat"], gate_center=-17.5, gate_scale=4.0,
    lr_sun_neg=lr_sun_neg, lr_sun_pos=lr_sun_pos,
    sun_gate_neg_center=-65.0, sun_gate_neg_scale=7.0,
    sun_gate_pos_center=6.0, sun_gate_pos_scale=10.0,
)

joblib.dump({"pipeline": model, "featureslow": featureslow}, MODEL_OUT)
print(f"Saved -> {MODEL_OUT}  ({len(X):,} training rows)")

reloaded = joblib.load(MODEL_OUT)
p = reloaded["pipeline"].predict_proba(X.iloc[[0]])[:, 1][0]
print(f"Reload sanity check: {p:.4f}")

Saved -> aurora_ebm_model_final_deployment.joblib  (22,174 training rows)
Reload sanity check: 0.0865
